In [3]:
import csv
import sys
sys.path.insert(0, ".")

import requests
from scrape_suppliers import (
    CSV_PATH, OUT_DIR, DELAY, HEADERS,
    load_db_lookup, load_overview_urls,
)

lookup, by_supplier = load_db_lookup()

with open(CSV_PATH) as f:
    rows = list(csv.DictReader(f))

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(lookup)} supplier-product pairs in DB | {len(rows)} links in CSV")

1633 supplier-product pairs in DB | 10 links in CSV


# Scrape Supplier Product Pages

Fetches and saves HTML for supplier product pages into `data/supplier_products/`.

Two modes — run either or both:
1. **CSV-direct** — fetches each URL from `data/fully_sourced_products_with_links.csv` directly using the exact supplier + raw material mapping. No crawling or fuzzy matching.
2. **Catalog crawl** — reads overview/collection URLs from `data/supplier_products_overview_pages.txt`, enumerates all product pages per supplier, and fuzzy-matches page titles to DB products by ingredient name.

Saved HTML is cleaned via `extract_content()` in `scrape_suppliers.py`: boilerplate tags (scripts, nav, footer, etc.) are stripped and the most product-relevant section is extracted before falling back to the full body.

In [4]:
import re, time, requests
from bs4 import BeautifulSoup
from scrape_suppliers import HEADERS, DELAY, OUT_DIR, extract_content

session = requests.Session()
print(f"Scraping {len(rows)} links from CSV...")

for row in rows:
    supplier = row["Supplier"]
    sku      = row["RawMaterial"]
    url      = row["Supplier_Product_Link"]

    key = (supplier, sku)
    if key not in lookup:
        print(f"  SKIP (not in DB): {supplier} / {sku}")
        continue

    supplier_id, product_id = lookup[key]
    fname    = f"{supplier_id}_{re.sub(r'[^\w\-]', '_', supplier)}_{product_id}_{sku}.html"
    out_path = OUT_DIR / fname

    try:
        resp    = session.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        soup    = BeautifulSoup(resp.text, "html.parser")
        content = extract_content(soup)
        out_path.write_text(content, encoding="utf-8")
        title   = soup.find("title")
        print(f"  SAVED: {fname}  ← {title.get_text(strip=True)[:70] if title else url}")
    except Exception as e:
        print(f"  ERROR: {url} → {e}")

    time.sleep(DELAY)

print("Done.")

Scraping 10 links from CSV...
  SAVED: 28_PureBulk_961_RM-C60-magnesium-glycinate-37b966a0.html  ← Search: 19 results found for "magnesium glycinate" – PureBulk, Inc.
  SAVED: 9_Capsuline_965_RM-C60-vegan-capsule-hypromellose-e7b625d7.html  ← Search: 4 results found for "vegan capsule hypromellose"  – Capsuline
  SAVED: 24_Nutra_Blend_885_RM-C57-cultured-nutrients-5c87fee4.html  ← Search: 2 results found for "cultured nutrients"
 – FeedsForLess.com
  SAVED: 24_Nutra_Blend_886_RM-C57-fruit-nutrients-b65036d4.html  ← Search: 0 results found for "fruit nutrients"
 – FeedsForLess.com
  SAVED: 24_Nutra_Blend_906_RM-C57-vegetable-nutrients-4667fd2d.html  ← Search: 0 results found for "vegetable nutrients"
 – FeedsForLess.com
  SAVED: 28_PureBulk_911_RM-C57-vitamin-d3-9213116f.html  ← Search: 21 results found for "vitamin d3" – PureBulk, Inc.
  SAVED: 28_PureBulk_281_RM-C13-b-vitamins-00134c85.html  ← Search: 55 results found for "b vitamins" – PureBulk, Inc.
  SAVED: 24_Nutra_Blend_282_RM-C1

## Mode 2: Catalog Crawl (alternative to CSV-direct above)

Instead of using known URLs from the CSV, this crawls each supplier's catalog page, enumerates all product URLs, and fuzzy-matches page titles to DB products. Use this when direct links aren't available.

In [ ]:
# import re, time, requests
# from bs4 import BeautifulSoup
# from scrape_suppliers import (
#     fetch, best_match, get_page_title, extract_content,
#     product_urls_shopify, product_urls_generic,
#     load_overview_urls, OUT_DIR, DELAY,
# )

# session = requests.Session()
# overview_urls = load_overview_urls()
# print(f"Overview pages: {list(overview_urls.keys())}\n")

# for supplier_name, catalog_url in overview_urls.items():
#     products = by_supplier.get(supplier_name, [])
#     if not products:
#         print(f"Skipping {supplier_name}: not in DB")
#         continue

#     sid = products[0]["SupplierId"]
#     saved_pids = {int(f.name.split("_")[-2]) for f in OUT_DIR.glob(f"{sid}_*_*_*.html")}
#     remaining  = [p for p in products if p["ProductId"] not in saved_pids]
#     if not remaining:
#         print(f"{supplier_name}: all products already saved, skipping")
#         continue

#     print(f"\n{'='*60}\n{supplier_name}  ({len(remaining)} remaining)")
#     get_urls = product_urls_shopify if "/collections/" in catalog_url else product_urls_generic
#     product_urls = get_urls(catalog_url, session)
#     print(f"  {len(product_urls)} product URLs found")

#     matched: set[int] = set()
#     for url in product_urls:
#         time.sleep(DELAY)
#         resp = fetch(url, session)
#         if not resp:
#             continue
#         soup  = BeautifulSoup(resp.text, "html.parser")
#         title = get_page_title(soup)
#         if not title:
#             continue

#         candidates = [p for p in remaining if p["ProductId"] not in matched]
#         if not candidates:
#             break
#         match = best_match(title, candidates)
#         if not match:
#             continue

#         matched.add(match["ProductId"])
#         fname   = f"{sid}_{re.sub(r'[^\w\-]', '_', supplier_name)}_{match['ProductId']}_{match['SKU']}.html"
#         content = extract_content(soup)
#         (OUT_DIR / fname).write_text(content, encoding="utf-8")
#         print(f"  SAVED: {fname}  ← '{title}'")

#     print(f"  saved {len(matched)}, unmatched: {len([p for p in remaining if p['ProductId'] not in matched])}")